In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import open3d as o3d
import torch

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
camK = np.load(r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes\scene_0000\realsense\camK.npy')
camK

array([[927.16973877,   0.        , 651.31506348],
       [  0.        , 927.36688232, 349.62133789],
       [  0.        ,   0.        ,   1.        ]])

In [3]:
fx = camK[0, 0]
fy = camK[1, 1]
cx = camK[0, 2]
cy = camK[1, 2]
print(fx, ",", fy,",", cx,",", cy)

927.1697387695312 , 927.3668823242188 , 651.3150634765625 , 349.621337890625


In [4]:
color_img = cv2.imread(r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes\scene_0000\realsense\rgb\0000.png')
color_img

array([[[ 94, 105,   8],
        [ 94, 105,   8],
        [ 92, 101,  16],
        ...,
        [ 80,  95,   6],
        [ 81,  96,   7],
        [ 82,  97,   8]],

       [[ 92, 102,   8],
        [ 93, 103,   9],
        [ 93, 101,  13],
        ...,
        [ 80,  94,   8],
        [ 81,  96,   7],
        [ 81,  96,   7]],

       [[ 94, 103,  11],
        [ 94, 103,  11],
        [ 93, 100,  15],
        ...,
        [ 84,  95,  10],
        [ 81,  95,   9],
        [ 81,  95,   9]],

       ...,

       [[135, 169, 146],
        [135, 169, 146],
        [134, 167, 147],
        ...,
        [104, 107,  99],
        [104, 107,  99],
        [104, 107,  99]],

       [[135, 169, 146],
        [135, 169, 146],
        [134, 167, 147],
        ...,
        [105, 106,  98],
        [105, 109,  98],
        [108, 112, 101]],

       [[132, 167, 142],
        [131, 166, 141],
        [128, 162, 139],
        ...,
        [102, 106,  95],
        [104, 108,  97],
        [107, 111, 100]]

In [5]:
color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)


In [6]:
depth_img = cv2.imread(r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes\scene_0000\realsense\depth\0000.png', cv2.IMREAD_UNCHANGED).astype(np.float32)
depth_img

array([[444., 444., 444., ..., 443., 443., 443.],
       [444., 444., 444., ..., 443., 443., 443.],
       [444., 444., 444., ..., 444., 444., 444.],
       ...,
       [532., 532., 532., ..., 371., 371., 371.],
       [532., 532., 532., ..., 371., 371., 371.],
       [532., 532., 532., ...,   0.,   0., 371.]], dtype=float32)

In [7]:
height, width = depth_img.shape
print(height, width)

720 1280


In [8]:
u_indicies = np.arange(width)
v_indices = np.arange(height)

In [9]:
u_grid, v_grid = np.meshgrid(u_indicies, v_indices)

In [10]:
z = depth_img / 1000
z

array([[0.444, 0.444, 0.444, ..., 0.443, 0.443, 0.443],
       [0.444, 0.444, 0.444, ..., 0.443, 0.443, 0.443],
       [0.444, 0.444, 0.444, ..., 0.444, 0.444, 0.444],
       ...,
       [0.532, 0.532, 0.532, ..., 0.371, 0.371, 0.371],
       [0.532, 0.532, 0.532, ..., 0.371, 0.371, 0.371],
       [0.532, 0.532, 0.532, ..., 0.   , 0.   , 0.371]], dtype=float32)

In [11]:
mask = z > 0
mask

array([[ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       ...,
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ...,  True,  True,  True],
       [ True,  True,  True, ..., False, False,  True]])

In [12]:
z_valid = z[mask]
z_valid

array([0.444, 0.444, 0.444, ..., 0.79 , 0.79 , 0.371], dtype=float32)

In [13]:
u_valid = u_grid[mask]
u_valid

array([   0,    1,    2, ..., 1242, 1243, 1279])

In [14]:
v_valid = v_grid[mask]
v_valid

array([  0,   0,   0, ..., 719, 719, 719])

In [15]:
color_valid = color_img[mask]
color_valid

array([[  8, 105,  94],
       [  8, 105,  94],
       [ 16, 101,  92],
       ...,
       [ 89, 102, 101],
       [ 95, 108, 107],
       [100, 111, 107]], dtype=uint8)

In [16]:
x_valid = (u_valid - cx) * z_valid / fx
y_valid = (v_valid - cy) * z_valid / fy

print(x_valid)
print(y_valid)

[-0.31189962 -0.31142075 -0.31094187 ...  0.50329631  0.50414837
  0.2511634 ]
[-0.16738993 -0.16738993 -0.16738993 ...  0.31466419  0.31466419
  0.14777267]


In [17]:
points_3d = np.stack((x_valid, y_valid, z_valid), axis=-1)
points_3d

array([[-0.31189962, -0.16738993,  0.44400001],
       [-0.31142075, -0.16738993,  0.44400001],
       [-0.31094187, -0.16738993,  0.44400001],
       ...,
       [ 0.50329631,  0.31466419,  0.79000002],
       [ 0.50414837,  0.31466419,  0.79000002],
       [ 0.2511634 ,  0.14777267,  0.37099999]])

<h1>Code Hoan Chinh


In [18]:
# =========================
# Load images
# =========================

depth_img = cv2.imread(r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes\scene_0000\realsense\depth\0000.png', cv2.IMREAD_UNCHANGED)
rgb_img = cv2.imread(r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes\scene_0000\realsense\rgb\0000.png')

# =========================
# Camera intrinsics
# =========================

fx = 927.17
fy = 927.37
cx = 651.32
cy = 349.62

# =========================
# Create pixel coordinate grid
# =========================

height, width = depth_img.shape

u = np.arange(width)
v = np.arange(height)

u_grid, v_grid = np.meshgrid(u, v)

# =========================
# Depth -> meters
# =========================

z = depth_img.astype(np.float32) / 1000.0

# =========================
# Remove invalid depth
# =========================

mask = z > 0

z_valid = z[mask]
u_valid = u_grid[mask]
v_valid = v_grid[mask]

# =========================
# Back-project to 3D
# =========================

x_valid = (u_valid - cx) * z_valid / fx
y_valid = (v_valid - cy) * z_valid / fy

# =========================
# Create point cloud array
# shape = (N, 3)
# =========================

points_3d = np.stack(
    (x_valid, y_valid, z_valid),
    axis=-1
)

# =========================
# RGB colors
# =========================

rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB)

colors = rgb_img.reshape(-1, 3) / 255.0
colors_valid = colors[mask.reshape(-1)]

# =========================
# Open3D Visualization
# =========================

pcd = o3d.geometry.PointCloud()

pcd.points = o3d.utility.Vector3dVector(points_3d)
pcd.colors = o3d.utility.Vector3dVector(colors_valid)

# =========================
# Downsample to 20k points
# =========================

target_points = 64000

num_points = len(points_3d)

ratio = target_points / num_points

ratio = min(ratio, 1.0)

pcd_down = pcd.random_down_sample(ratio)

print("Original points:", num_points)
print("Downsampled points:", len(pcd_down.points))

# =========================
# Visualize
# =========================

o3d.visualization.draw_geometries(
    [pcd_down],
    window_name="Point Cloud",
    width=1280,
    height=720
)

Original points: 892215
Downsampled points: 64000


<h1>Thuc hien tren toan bo tap du lieu

In [19]:
import torch

device = torch.device('cuda')

print(torch.cuda.get_device_name(0))


NVIDIA GeForce RTX 5050 Laptop GPU


In [ ]:
# import os
# import cv2
# import numpy as np
# import torch
# import open3d as o3d
# from tqdm import tqdm

# # =========================
# # CUDA
# # =========================

# device = torch.device('cuda')

# print(torch.cuda.get_device_name(0))

# # =========================
# # Dataset paths
# # =========================

# dataset_root = r'E:\Users\VsCode\Computer Vision Project\grasp_dataset\scenes'

# output_root = r'E:\Users\VsCode\point_cloud_dataset'

# os.makedirs(output_root, exist_ok=True)

# scene_list = sorted(os.listdir(dataset_root))

# # =========================
# # Process scenes
# # =========================

# for scene_name in tqdm(scene_list):

#     scene_path = os.path.join(
#         dataset_root,
#         scene_name,
#         'realsense'
#     )

#     depth_dir = os.path.join(scene_path, 'depth')
#     rgb_dir = os.path.join(scene_path, 'rgb')

#     camK_path = os.path.join(
#         scene_path,
#         'camK.npy'
#     )

#     # =========================
#     # Camera intrinsics
#     # =========================

#     camK = np.load(camK_path)

#     fx = camK[0, 0]
#     fy = camK[1, 1]

#     cx = camK[0, 2]
#     cy = camK[1, 2]

#     # =========================
#     # Output folder
#     # =========================

#     output_scene_dir = os.path.join(
#         output_root,
#         scene_name
#     )

#     os.makedirs(
#         output_scene_dir,
#         exist_ok=True
#     )

#     depth_files = sorted(
#         os.listdir(depth_dir)
#     )

#     # =========================
#     # Process frames
#     # =========================

#     for file_name in depth_files:

#         depth_path = os.path.join(
#             depth_dir,
#             file_name
#         )

#         rgb_path = os.path.join(
#             rgb_dir,
#             file_name
#         )

#         # =========================
#         # Load images
#         # =========================

#         depth_img = cv2.imread(
#             depth_path,
#             cv2.IMREAD_UNCHANGED
#         )

#         rgb_img = cv2.imread(rgb_path)

#         if depth_img is None or rgb_img is None:
#             continue

#         # =========================
#         # RGB -> RGB
#         # =========================

#         rgb_img = cv2.cvtColor(
#             rgb_img,
#             cv2.COLOR_BGR2RGB
#         )

#         # =========================
#         # Move to GPU
#         # =========================

#         depth_tensor = torch.from_numpy(
#             depth_img.astype(np.float32)
#         ).to(device)

#         rgb_tensor = torch.from_numpy(
#             rgb_img.astype(np.float32) / 255.0
#         ).to(device)

#         # =========================
#         # Depth -> meters
#         # =========================

#         z = depth_tensor / 1000.0

#         height, width = z.shape

#         # =========================
#         # Pixel grid
#         # =========================

#         u = torch.arange(
#             width,
#             device=device
#         )

#         v = torch.arange(
#             height,
#             device=device
#         )

#         u_grid, v_grid = torch.meshgrid(
#             u,
#             v,
#             indexing='xy'
#         )

#         # =========================
#         # Valid depth mask
#         # =========================

#         mask = z > 0

#         z_valid = z[mask]

#         u_valid = u_grid[mask]
#         v_valid = v_grid[mask]

#         # =========================
#         # Back-project
#         # =========================

#         x_valid = (
#             (u_valid - cx)
#             * z_valid
#             / fx
#         )

#         y_valid = (
#             (v_valid - cy)
#             * z_valid
#             / fy
#         )

#         # =========================
#         # Point cloud tensor
#         # =========================

#         points_3d = torch.stack(
#             [
#                 x_valid,
#                 y_valid,
#                 z_valid
#             ],
#             dim=-1
#         )

#         # =========================
#         # RGB colors
#         # =========================

#         colors_valid = rgb_tensor[
#             mask
#         ]

#         # =========================
#         # Downsample to 20k
#         # =========================

#         target_points = 20000

#         num_points = points_3d.shape[0]

#         if num_points > target_points:

#             indices = torch.randperm(
#                 num_points,
#                 device=device
#             )[:target_points]

#             points_3d = points_3d[indices]

#             colors_valid = colors_valid[indices]

#         # =========================
#         # Move back to CPU
#         # =========================

#         points_np = points_3d.cpu().numpy()

#         colors_np = colors_valid.cpu().numpy()

#         # =========================
#         # Open3D point cloud
#         # =========================

#         pcd = o3d.geometry.PointCloud()

#         pcd.points = o3d.utility.Vector3dVector(
#             points_np
#         )

#         pcd.colors = o3d.utility.Vector3dVector(
#             colors_np
#         )

#         # =========================
#         # Save .ply
#         # =========================

#         save_path = os.path.join(
#             output_scene_dir,
#             file_name.replace('.png', '.ply')
#         )

#         o3d.io.write_point_cloud(
#             save_path,
#             pcd
#         )

# print("Done.")

NVIDIA GeForce RTX 5050 Laptop GPU


100%|██████████| 190/190 [2:06:50<00:00, 40.06s/it]  

Done.
